# INF2006 G12 – Task 2 pipeline driver (Bash kernel)

One notebook that runs the whole Task 2 pipeline for **one profile, one configuration and one repetition**, on a fresh
Hive database and fresh HDFS output folders, and records everything the performance section asks for.

| Stage (timed) | Engine | What it does |
|---|---|---|
| `gdelt_streaming` | Hadoop Streaming (map-only) | Reads the staged GDELT zips from HDFS inside parallel map tasks, validates structure, projects 19 columns, writes tagged gzip text |
| `hive_raw_tables` | Hive | Creates the run database and external tables over the staged prices and the Streaming output |
| `hive_clean_prices` | Hive | Validates prices; writes cleaned / rejected / out-of-scope tables (your existing rules) |
| `hive_price_splits` | Hive | Chronological splits 2013-2015 / 2016 / 2017 |
| `hive_clean_gdelt` | Hive | Value-level GDELT validation, cleaned + rejected tables, daily features by `DATEADDED` |
| `hive_reconcile` | Hive | Row-count reconciliation (`PASS`/`FAIL`) for prices and GDELT |
| `spark_features` | PySpark | Rolling features, market-wide and lagged GDELT features, targets, split + boundary rules, Hive-registered output tables |
| `spark_validate` | PySpark | Re-checks dates, boundaries, duplicates, leakage columns, non-finite values, row accounting |
| `train` (optional) | any | Set `TRAIN_CMD` so Task 3 training is inside the timed total |

**Not timed (reported separately):** cluster start, dependency set-up, S3→HDFS staging (Task 1 notebook), writing the helper
files, discovering inputs, building the development profile inputs, summaries and clean-up.

**Run it interactively** (profile `dev`, `reference`, rep 1) with *Run All*, or **unattended** with
`PROFILE=full CONFIG=alt REP=2 jupyter nbconvert --to notebook --execute INF2006_G12_task2_driver.ipynb --ExecutePreprocessor.timeout=-1`
(see `run_matrix.sh` for the full 3-repetition matrix). The executed copy is your dated EMR evidence.

**Fresh state per run:** new Hive database, new HDFS output folder, new Spark applications, no `cache()` anywhere.

## 1. Parameters (override with environment variables)

In [1]:
# ---------------- parameters ----------------
export PROFILE=${PROFILE:-dev}                 # dev | full
export CONFIG=${CONFIG:-reference}             # reference | alt
export REP=${REP:-1}                           # repetition number (1, 2, 3)
export CORPUS=${CORPUS:-/corpus}               # HDFS root created by the staging notebook
export RAW_PRICES_ROOT=${RAW_PRICES_ROOT:-$CORPUS/raw/prices}
export RAW_GDELT_DIR=${RAW_GDELT_DIR:-$CORPUS/raw/gdelt}
export RAW_PRICES_FILE=${RAW_PRICES_FILE:-}    # leave empty to auto-discover the single price CSV
export DEV_TICKERS=${DEV_TICKERS:-50}          # development profile: first N tickers (alphabetical)
export DEV_GDELT_ARCHIVES=${DEV_GDELT_ARCHIVES:-14}   # development profile: first N daily archives
export USE_2018_LABELS=${USE_2018_LABELS:-0}   # 1 = label late-2017 rows from 2018 prices (outside the brief's window)
export KEEP_BULK=${KEEP_BULK:-0}               # 1 = keep the large intermediate HDFS folders after the run
export SAMPLE_SECS=${SAMPLE_SECS:-30}          # resource sampling interval
export BUCKET=${BUCKET:-}                      # optional S3 bucket for evidence upload (no credentials here)
export TRAIN_CMD=${TRAIN_CMD:-}                # optional: command that trains the Task 3 model (timed as stage "train")
export DRIVER_HOME=${DRIVER_HOME:-$HOME/task2_driver}
export RUNS_ROOT=${RUNS_ROOT:-$HOME/task2_runs}
export STREAMING_JAR=${STREAMING_JAR:-$(ls /usr/lib/hadoop/hadoop-streaming*.jar 2>/dev/null | head -1)}
export RUN_ID=${RUN_ID:-$(date -u +%Y%m%dT%H%M%SZ)}

# ---------------- configurations: change ONLY software settings; hardware, release, replication fixed ----------------
case "$CONFIG" in
  reference) SHUFFLE_PARTS=200; LINES_PER_MAP=25 ;;   # brief's reference: Spark default shuffle partitions
  alt)       SHUFFLE_PARTS=16;  LINES_PER_MAP=12 ;;   # alternative: partitions sized to 4 executor cores, more map tasks
  *) echo "CONFIG must be reference or alt"; CONFIG=invalid ;;
esac
SPARK_ARGS=(--master yarn --deploy-mode client
            --conf spark.dynamicAllocation.enabled=false
            --num-executors 2 --executor-cores 2 --executor-memory 6g
            --conf spark.executor.memoryOverhead=2g --driver-memory 2g
            --conf spark.sql.shuffle.partitions=$SHUFFLE_PARTS)

RUN_NAME=${PROFILE}_${CONFIG}_rep${REP}_${RUN_ID}
OUT=$CORPUS/runs/$RUN_NAME                     # fresh HDFS output location for this run
LOCAL_RUN=$RUNS_ROOT/$RUN_NAME                 # local logs and evidence for this run
DB=$(echo "g12_${PROFILE}_${CONFIG}_r${REP}_${RUN_ID}" | tr 'A-Z' 'a-z')   # fresh Hive database
PROFILE_DIR=$CORPUS/profiles/$PROFILE          # frozen input profile (built once, not timed)
ASSETS=$DRIVER_HOME/assets
export RUN_NAME OUT LOCAL_RUN DB PROFILE_DIR ASSETS SHUFFLE_PARTS LINES_PER_MAP
mkdir -p "$LOCAL_RUN/logs" "$ASSETS/sql"
FAILED=
echo "run=$RUN_NAME"; echo "db=$DB"; echo "hdfs out=$OUT"; echo "local evidence=$LOCAL_RUN"
echo "spark: ${SPARK_ARGS[*]}"; echo "streaming lines per map=$LINES_PER_MAP"

run=full_reference_rep2_20261120T114124Z


db=g12_full_reference_r2_20261120t114124z


hdfs out=/corpus/runs/full_reference_rep2_20261120T114124Z


local evidence=/home/hadoop/task2_runs/full_reference_rep2_20261120T114124Z


spark: --master yarn --deploy-mode client --conf spark.dynamicAllocation.enabled=false --num-executors 2 --executor-cores 2 --executor-memory 6g --conf spark.executor.memoryOverhead=2g --driver-memory 2g --conf spark.sql.shuffle.partitions=200


streaming lines per map=25


## 2. Write the pipeline files (not timed)
The notebook writes every mapper, SQL and PySpark file it runs, so the notebook alone reproduces the pipeline. The same files are shipped in `assets/`.

In [2]:
# --- Hadoop Streaming mapper and the five Hive SQL files ---
cat > "$ASSETS/gdelt_zip_mapper.py" <<'__ASSET_EOF__'
#!/usr/bin/env python3
"""Hadoop Streaming mapper (map-only) for the Task 2 GDELT stage.

Input : lines holding HDFS paths of GDELT 1.0 daily archives (*.export.CSV.zip),
        delivered by NLineInputFormat. Each archive is read straight from HDFS
        inside the map task, so unzipping, structural validation and projection
        all run in parallel on the worker nodes.
Output: tab-separated, tagged lines (the Hive table gdelt_stream reads them):
  V <19 projected fields>                   valid event row
  R <reason> <archive> <raw line>           rejected row (tabs in the raw line -> 0x1f)
  A <archive> <rows> <valid> <rejected> <zip_bytes>   one audit line per archive

Structural rules applied here (value-level rules are applied later in Hive):
  - exactly GDELT_COLS (58) tab-separated fields        else wrong_column_count:<n>
  - valid UTF-8                                          else invalid_utf8
  - GLOBALEVENTID not repeated inside the same archive   else duplicate_event_id_in_archive
  - archive must be a readable zip                       else unreadable_archive
Duplicates across archives are not detected here.

Projected fields (0-based GDELT 1.0 indexes):
  0 GLOBALEVENTID, 1 SQLDATE, 26 EventCode, 27 EventBaseCode, 28 EventRootCode,
  29 QuadClass, 30 GoldsteinScale, 31 NumMentions, 32 NumSources, 33 NumArticles,
  34 AvgTone, 6 Actor1Name, 7 Actor1CountryCode, 16 Actor2Name, 17 Actor2CountryCode,
  51 ActionGeo_CountryCode, 50 ActionGeo_FullName, 56 DATEADDED, 57 SOURCEURL
"""
import io
import os
import shlex
import subprocess
import sys
import zipfile

HDFS_CAT = os.environ.get("HDFS_CAT", "hdfs dfs -cat")
EXPECTED_COLS = int(os.environ.get("GDELT_COLS", "58"))
KEEP = [0, 1, 26, 27, 28, 29, 30, 31, 32, 33, 34, 6, 7, 16, 17, 51, 50, 56, 57]
MAX_RAW = 1500


def emit(*fields):
    sys.stdout.write("\t".join(str(f) for f in fields) + "\n")


def read_zip(path):
    p = subprocess.run(shlex.split(HDFS_CAT) + [path],
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if p.returncode != 0:
        raise IOError(p.stderr.decode("utf-8", "replace")[:150])
    return p.stdout


def reject(reason, name, raw):
    raw = raw[:MAX_RAW].replace("\t", "\x1f").replace("\r", "").replace("\n", " ")
    emit("R", reason, name, raw)


for line_in in sys.stdin:
    path = line_in.rstrip("\n").split("\t")[-1].strip()
    if not path:
        continue
    name = os.path.basename(path)
    sys.stderr.write("reporter:status:" + name + "\n")
    rows = valid = rejected = 0
    nbytes = 0
    try:
        data = read_zip(path)
        nbytes = len(data)
        zf = zipfile.ZipFile(io.BytesIO(data))
        seen = set()
        for member in zf.namelist():
            with zf.open(member) as fh:
                for b in fh:
                    rows += 1
                    try:
                        text = b.decode("utf-8").rstrip("\r\n")
                    except UnicodeDecodeError:
                        rejected += 1
                        reject("invalid_utf8", name, b.decode("utf-8", "replace"))
                        continue
                    f = text.split("\t")
                    if len(f) != EXPECTED_COLS:
                        rejected += 1
                        reject("wrong_column_count:%d" % len(f), name, text)
                        continue
                    eid = f[0]
                    if eid.isdigit():
                        if eid in seen:
                            rejected += 1
                            reject("duplicate_event_id_in_archive", name, text)
                            continue
                        seen.add(eid)
                    valid += 1
                    emit("V", *[f[i] for i in KEEP])
    except Exception as e:
        rejected += 1
        reject("unreadable_archive", name, str(e))
        sys.stderr.write("reporter:counter:GDELT,unreadable_archives,1\n")
    emit("A", name, rows, valid, rejected, nbytes)
    sys.stderr.write("reporter:counter:GDELT,rows_read,%d\n" % rows)
    sys.stderr.write("reporter:counter:GDELT,valid_rows,%d\n" % valid)
    sys.stderr.write("reporter:counter:GDELT,rejected_rows,%d\n" % rejected)
    sys.stderr.write("reporter:counter:GDELT,archives,1\n")
__ASSET_EOF__
cat > "$ASSETS/sql/01_create_db_raw_tables.sql" <<'__ASSET_EOF__'
-- Task 2 / stage hive_raw_tables
-- Variables (passed with --hivevar): DB, RAW_PRICES, STREAM_OUT
CREATE DATABASE IF NOT EXISTS ${hivevar:DB};
USE ${hivevar:DB};

-- Raw prices: original CSV text, every column kept as STRING so nothing is lost on read.
CREATE EXTERNAL TABLE raw_prices (
    `date` STRING,
    `open` STRING,
    `high` STRING,
    `low` STRING,
    `close` STRING,
    `volume` STRING,
    `name` STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION '${hivevar:RAW_PRICES}'
TBLPROPERTIES ('skip.header.line.count' = '1');

-- Output of the Hadoop Streaming GDELT job: tagged lines (V = valid, R = rejected, A = audit).
CREATE EXTERNAL TABLE gdelt_stream (
    tag STRING,
    c01 STRING, c02 STRING, c03 STRING, c04 STRING, c05 STRING,
    c06 STRING, c07 STRING, c08 STRING, c09 STRING, c10 STRING,
    c11 STRING, c12 STRING, c13 STRING, c14 STRING, c15 STRING,
    c16 STRING, c17 STRING, c18 STRING, c19 STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '\t'
STORED AS TEXTFILE
LOCATION '${hivevar:STREAM_OUT}';
__ASSET_EOF__
cat > "$ASSETS/sql/02_clean_prices.sql" <<'__ASSET_EOF__'
-- Task 2 / stage hive_clean_prices   (cleaning rules unchanged from the team's 02_clean_prices.sql)
-- Variables: DB, OUT
USE ${hivevar:DB};

CREATE VIEW price_parsed AS
SELECT
    `date`, `open`, `high`, `low`, `close`, `volume`, `name`,
    CAST(TRIM(`date`) AS DATE) AS trade_date,
    CAST(TRIM(`open`) AS DOUBLE) AS open_value,
    CAST(TRIM(`high`) AS DOUBLE) AS high_value,
    CAST(TRIM(`low`) AS DOUBLE) AS low_value,
    CAST(TRIM(`close`) AS DOUBLE) AS close_value,
    CAST(TRIM(`volume`) AS BIGINT) AS volume_value
FROM raw_prices;

CREATE VIEW price_validation AS
SELECT *,
    YEAR(trade_date) AS year,
    CASE
        WHEN COALESCE(TRIM(`date`), '') = ''
          OR COALESCE(TRIM(`open`), '') = ''
          OR COALESCE(TRIM(`high`), '') = ''
          OR COALESCE(TRIM(`low`), '') = ''
          OR COALESCE(TRIM(`close`), '') = ''
          OR COALESCE(TRIM(`volume`), '') = ''
          OR COALESCE(TRIM(`name`), '') = ''
            THEN 'null_field'
        WHEN trade_date IS NULL THEN 'invalid_date'
        WHEN open_value IS NULL THEN 'invalid_open'
        WHEN high_value IS NULL THEN 'invalid_high'
        WHEN low_value IS NULL THEN 'invalid_low'
        WHEN close_value IS NULL THEN 'invalid_close'
        WHEN volume_value IS NULL THEN 'invalid_volume'
        WHEN open_value <= 0 OR high_value <= 0
          OR low_value <= 0 OR close_value <= 0
            THEN 'non_positive_price'
        WHEN volume_value <= 0 THEN 'invalid_volume'
        WHEN high_value < low_value THEN 'high_lt_low'
        WHEN open_value < low_value OR open_value > high_value
            THEN 'open_outside_range'
        WHEN close_value < low_value OR close_value > high_value
            THEN 'close_outside_range'
        ELSE NULL
    END AS reject_reason
FROM price_parsed;

CREATE TABLE cleaned_prices (
    ticker STRING,
    trade_date DATE,
    open_price DOUBLE,
    high_price DOUBLE,
    low_price DOUBLE,
    close_price DOUBLE,
    volume BIGINT
)
PARTITIONED BY (year INT)
STORED AS PARQUET
LOCATION '${hivevar:OUT}/cleaned/prices';

CREATE TABLE rejected_prices (
    `date` STRING,
    `open` STRING,
    `high` STRING,
    `low` STRING,
    `close` STRING,
    `volume` STRING,
    `name` STRING,
    reject_reason STRING
)
STORED AS PARQUET
LOCATION '${hivevar:OUT}/rejected/prices';

CREATE TABLE out_of_scope_prices (
    `date` STRING,
    `open` STRING,
    `high` STRING,
    `low` STRING,
    `close` STRING,
    `volume` STRING,
    `name` STRING
)
STORED AS PARQUET
LOCATION '${hivevar:OUT}/out_of_scope/prices';

INSERT OVERWRITE TABLE out_of_scope_prices
SELECT `date`, `open`, `high`, `low`, `close`, `volume`, `name`
FROM price_validation
WHERE trade_date > CAST('2017-12-31' AS DATE);

INSERT OVERWRITE TABLE rejected_prices
SELECT `date`, `open`, `high`, `low`, `close`, `volume`, `name`, reject_reason
FROM price_validation
WHERE reject_reason IS NOT NULL
  AND (trade_date IS NULL OR trade_date <= CAST('2017-12-31' AS DATE));

SET hive.exec.dynamic.partition=true;
SET hive.exec.dynamic.partition.mode=nonstrict;

INSERT OVERWRITE TABLE cleaned_prices PARTITION (year)
SELECT
    name, trade_date, open_value, high_value,
    low_value, close_value, volume_value, year
FROM price_validation
WHERE trade_date BETWEEN CAST('2013-01-01' AS DATE)
                     AND CAST('2017-12-31' AS DATE)
  AND reject_reason IS NULL;
__ASSET_EOF__
cat > "$ASSETS/sql/03_price_splits.sql" <<'__ASSET_EOF__'
-- Task 2 / stage hive_price_splits   (chronological splits unchanged from the team's 03_price_splits.sql)
-- Variables: DB, OUT
USE ${hivevar:DB};

CREATE TABLE training_prices (
    ticker STRING, trade_date DATE, open_price DOUBLE, high_price DOUBLE,
    low_price DOUBLE, close_price DOUBLE, volume BIGINT
)
STORED AS PARQUET
LOCATION '${hivevar:OUT}/splits/prices/training';

CREATE TABLE validation_prices (
    ticker STRING, trade_date DATE, open_price DOUBLE, high_price DOUBLE,
    low_price DOUBLE, close_price DOUBLE, volume BIGINT
)
STORED AS PARQUET
LOCATION '${hivevar:OUT}/splits/prices/validation';

CREATE TABLE testing_prices (
    ticker STRING, trade_date DATE, open_price DOUBLE, high_price DOUBLE,
    low_price DOUBLE, close_price DOUBLE, volume BIGINT
)
STORED AS PARQUET
LOCATION '${hivevar:OUT}/splits/prices/testing';

INSERT OVERWRITE TABLE training_prices
SELECT ticker, trade_date, open_price, high_price, low_price, close_price, volume
FROM cleaned_prices
WHERE trade_date BETWEEN '2013-01-01' AND '2015-12-31';

INSERT OVERWRITE TABLE validation_prices
SELECT ticker, trade_date, open_price, high_price, low_price, close_price, volume
FROM cleaned_prices
WHERE trade_date BETWEEN '2016-01-01' AND '2016-12-31';

INSERT OVERWRITE TABLE testing_prices
SELECT ticker, trade_date, open_price, high_price, low_price, close_price, volume
FROM cleaned_prices
WHERE trade_date BETWEEN '2017-01-01' AND '2017-12-31';
__ASSET_EOF__
cat > "$ASSETS/sql/04_clean_gdelt.sql" <<'__ASSET_EOF__'
-- Task 2 / stage hive_clean_gdelt   (value rules unchanged from the team's 05_clean_gdelt_final.sql)
-- Variables: DB, OUT
USE ${hivevar:DB};

-- 0. Split the tagged Streaming output into valid events, structural rejects and per-archive audit lines.
CREATE VIEW gdelt_valid_raw AS
SELECT c01 AS GLOBALEVENTID, c02 AS SQLDATE, c03 AS EventCode, c04 AS EventBaseCode,
       c05 AS EventRootCode, c06 AS QuadClass, c07 AS GoldsteinScale, c08 AS NumMentions,
       c09 AS NumSources, c10 AS NumArticles, c11 AS AvgTone, c12 AS Actor1Name,
       c13 AS Actor1CountryCode, c14 AS Actor2Name, c15 AS Actor2CountryCode,
       c16 AS ActionGeo_CountryCode, c17 AS ActionGeo_FullName, c18 AS DATEADDED,
       c19 AS SOURCEURL
FROM gdelt_stream WHERE tag = 'V';

CREATE VIEW gdelt_struct_rejects AS
SELECT c01 AS reject_reason, c02 AS source_archive, c03 AS raw_line
FROM gdelt_stream WHERE tag = 'R';

CREATE VIEW gdelt_audit AS
SELECT c01 AS source_archive,
       CAST(c02 AS BIGINT) AS rows_read,
       CAST(c03 AS BIGINT) AS valid_rows,
       CAST(c04 AS BIGINT) AS rejected_rows,
       CAST(c05 AS BIGINT) AS zip_bytes
FROM gdelt_stream WHERE tag = 'A';

-- 1. Parse fields into typed columns.
CREATE VIEW gdelt_parsed AS
SELECT
    TRIM(GLOBALEVENTID) AS event_id,
    CASE
        WHEN TRIM(SQLDATE) RLIKE '^[0-9]{8}$'
        THEN CAST(CONCAT(SUBSTR(TRIM(SQLDATE), 1, 4), '-',
                         SUBSTR(TRIM(SQLDATE), 5, 2), '-',
                         SUBSTR(TRIM(SQLDATE), 7, 2)) AS DATE)
        ELSE NULL
    END AS event_date,
    CASE
        WHEN TRIM(DATEADDED) RLIKE '^[0-9]{8}$'
        THEN CAST(CONCAT(SUBSTR(TRIM(DATEADDED), 1, 4), '-',
                         SUBSTR(TRIM(DATEADDED), 5, 2), '-',
                         SUBSTR(TRIM(DATEADDED), 7, 2)) AS DATE)
        ELSE NULL
    END AS available_date,
    TRIM(EventCode) AS event_code,
    TRIM(EventBaseCode) AS event_base_code,
    TRIM(EventRootCode) AS event_root_code,
    CAST(TRIM(QuadClass) AS INT) AS quad_class,
    CAST(TRIM(GoldsteinScale) AS DOUBLE) AS goldstein_scale,
    CAST(TRIM(NumMentions) AS INT) AS num_mentions,
    CAST(TRIM(NumSources) AS INT) AS num_sources,
    CAST(TRIM(NumArticles) AS INT) AS num_articles,
    CAST(TRIM(AvgTone) AS DOUBLE) AS avg_tone,
    Actor1Name AS actor1_name,
    Actor1CountryCode AS actor1_country,
    Actor2Name AS actor2_name,
    Actor2CountryCode AS actor2_country,
    ActionGeo_CountryCode AS action_country,
    ActionGeo_FullName AS action_location,
    TRIM(DATEADDED) AS date_added,
    SOURCEURL AS source_url,
    TRIM(QuadClass) AS raw_quad_class,
    TRIM(GoldsteinScale) AS raw_goldstein_scale,
    TRIM(NumMentions) AS raw_num_mentions,
    TRIM(NumSources) AS raw_num_sources,
    TRIM(NumArticles) AS raw_num_articles,
    TRIM(AvgTone) AS raw_avg_tone
FROM gdelt_valid_raw;

-- 2. Classify invalid records.
CREATE VIEW gdelt_classified AS
SELECT *,
    CASE
        WHEN event_id IS NULL OR event_id = '' THEN 'missing_event_id'
        WHEN event_id NOT RLIKE '^[0-9]+$' THEN 'invalid_event_id'
        WHEN event_date IS NULL THEN 'invalid_event_date'
        WHEN available_date IS NULL THEN 'invalid_available_date'
        WHEN event_code IS NULL OR event_code = '' THEN 'missing_event_code'
        WHEN quad_class IS NULL OR raw_quad_class NOT RLIKE '^[1-4]$' THEN 'invalid_quad_class'
        WHEN goldstein_scale IS NULL
          OR raw_goldstein_scale NOT RLIKE '^-?[0-9]+(\\.[0-9]+)?$' THEN 'invalid_goldstein_scale'
        WHEN num_mentions IS NULL OR raw_num_mentions NOT RLIKE '^[0-9]+$' THEN 'invalid_num_mentions'
        WHEN num_sources IS NULL OR raw_num_sources NOT RLIKE '^[0-9]+$' THEN 'invalid_num_sources'
        WHEN num_articles IS NULL OR raw_num_articles NOT RLIKE '^[0-9]+$' THEN 'invalid_num_articles'
        WHEN avg_tone IS NULL
          OR raw_avg_tone NOT RLIKE '^-?[0-9]+(\\.[0-9]+)?$' THEN 'invalid_avg_tone'
        ELSE NULL
    END AS reject_reason
FROM gdelt_parsed;

-- 3. Rejected records stay separate (value-level rejects, then structural rejects from Streaming).
CREATE TABLE rejected_gdelt (
    event_id STRING, event_date DATE, available_date DATE, event_code STRING, reject_reason STRING
)
STORED AS PARQUET
LOCATION '${hivevar:OUT}/rejected/gdelt';

CREATE TABLE rejected_gdelt_structural (
    reject_reason STRING, source_archive STRING, raw_line STRING
)
STORED AS PARQUET
LOCATION '${hivevar:OUT}/rejected/gdelt_structural';

-- 4. Cleaned events.
CREATE TABLE cleaned_gdelt_events (
    event_id STRING, event_date DATE, available_date DATE, event_code STRING,
    event_base_code STRING, event_root_code STRING, quad_class INT, goldstein_scale DOUBLE,
    num_mentions INT, num_sources INT, num_articles INT, avg_tone DOUBLE,
    actor1_name STRING, actor1_country STRING, actor2_name STRING, actor2_country STRING,
    action_country STRING, action_location STRING, date_added STRING, source_url STRING
)
STORED AS PARQUET
LOCATION '${hivevar:OUT}/cleaned/gdelt/events';

INSERT OVERWRITE TABLE rejected_gdelt
SELECT event_id, event_date, available_date, event_code, reject_reason
FROM gdelt_classified WHERE reject_reason IS NOT NULL;

INSERT OVERWRITE TABLE rejected_gdelt_structural
SELECT reject_reason, source_archive, raw_line FROM gdelt_struct_rejects;

INSERT OVERWRITE TABLE cleaned_gdelt_events
SELECT event_id, event_date, available_date, event_code, event_base_code, event_root_code,
       quad_class, goldstein_scale, num_mentions, num_sources, num_articles, avg_tone,
       actor1_name, actor1_country, actor2_name, actor2_country, action_country,
       action_location, date_added, source_url
FROM gdelt_classified WHERE reject_reason IS NULL;

-- 5. Daily news features by availability date (DATEADDED), never by event date.
CREATE TABLE daily_gdelt_features (
    available_date DATE, event_count BIGINT, avg_goldstein DOUBLE, avg_tone DOUBLE,
    total_mentions BIGINT, total_sources BIGINT, total_articles BIGINT, distinct_event_codes BIGINT
)
STORED AS PARQUET
LOCATION '${hivevar:OUT}/cleaned/gdelt/daily_features';

INSERT OVERWRITE TABLE daily_gdelt_features
SELECT available_date, COUNT(*), AVG(goldstein_scale), AVG(avg_tone),
       SUM(num_mentions), SUM(num_sources), SUM(num_articles), COUNT(DISTINCT event_code)
FROM cleaned_gdelt_events
GROUP BY available_date;
__ASSET_EOF__
cat > "$ASSETS/sql/05_reconcile.sql" <<'__ASSET_EOF__'
-- Task 2 / stage hive_reconcile   (run with `hive -S`; prints check<TAB>expected<TAB>actual<TAB>status)
-- Variables: DB, EXPECTED_ARCHIVES
USE ${hivevar:DB};

SELECT 'prices_raw_rows' AS check_name, CAST(NULL AS BIGINT) AS expected, COUNT(*) AS actual, 'INFO' AS status
FROM raw_prices
UNION ALL
SELECT 'prices_cleaned_rows', CAST(NULL AS BIGINT), COUNT(*), 'INFO' FROM cleaned_prices
UNION ALL
SELECT 'prices_rejected_rows', CAST(NULL AS BIGINT), COUNT(*), 'INFO' FROM rejected_prices
UNION ALL
SELECT 'prices_out_of_scope_rows', CAST(NULL AS BIGINT), COUNT(*), 'INFO' FROM out_of_scope_prices
UNION ALL
SELECT 'prices_raw_eq_cleaned_rejected_out_of_scope', r.n, c.n + x.n + o.n,
       CASE WHEN r.n = c.n + x.n + o.n THEN 'PASS' ELSE 'FAIL' END
FROM (SELECT COUNT(*) AS n FROM raw_prices) r
CROSS JOIN (SELECT COUNT(*) AS n FROM cleaned_prices) c
CROSS JOIN (SELECT COUNT(*) AS n FROM rejected_prices) x
CROSS JOIN (SELECT COUNT(*) AS n FROM out_of_scope_prices) o
UNION ALL
SELECT 'prices_cleaned_eq_train_val_test', c.n, t.n + v.n + s.n,
       CASE WHEN c.n = t.n + v.n + s.n THEN 'PASS' ELSE 'FAIL' END
FROM (SELECT COUNT(*) AS n FROM cleaned_prices) c
CROSS JOIN (SELECT COUNT(*) AS n FROM training_prices) t
CROSS JOIN (SELECT COUNT(*) AS n FROM validation_prices) v
CROSS JOIN (SELECT COUNT(*) AS n FROM testing_prices) s
UNION ALL
SELECT 'gdelt_archives_audited', CAST(${hivevar:EXPECTED_ARCHIVES} AS BIGINT), COUNT(*),
       CASE WHEN COUNT(*) = ${hivevar:EXPECTED_ARCHIVES} THEN 'PASS' ELSE 'FAIL' END
FROM gdelt_audit
UNION ALL
SELECT 'gdelt_rows_read', CAST(NULL AS BIGINT), SUM(rows_read), 'INFO' FROM gdelt_audit
UNION ALL
SELECT 'gdelt_streaming_rows_eq_valid_plus_rejected', SUM(rows_read), SUM(valid_rows) + SUM(rejected_rows),
       CASE WHEN SUM(rows_read) = SUM(valid_rows) + SUM(rejected_rows) THEN 'PASS' ELSE 'FAIL' END
FROM gdelt_audit
UNION ALL
SELECT 'gdelt_valid_eq_cleaned_plus_value_rejected', a.n, c.n + r.n,
       CASE WHEN a.n = c.n + r.n THEN 'PASS' ELSE 'FAIL' END
FROM (SELECT COALESCE(SUM(valid_rows), 0) AS n FROM gdelt_audit) a
CROSS JOIN (SELECT COUNT(*) AS n FROM cleaned_gdelt_events) c
CROSS JOIN (SELECT COUNT(*) AS n FROM rejected_gdelt) r
UNION ALL
SELECT 'gdelt_structural_rejects_eq_audit', a.n, s.n,
       CASE WHEN a.n = s.n THEN 'PASS' ELSE 'FAIL' END
FROM (SELECT COALESCE(SUM(rejected_rows), 0) AS n FROM gdelt_audit) a
CROSS JOIN (SELECT COUNT(*) AS n FROM rejected_gdelt_structural) s
UNION ALL
SELECT 'gdelt_unreadable_archives', CAST(0 AS BIGINT), COUNT(*),
       CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM rejected_gdelt_structural WHERE reject_reason = 'unreadable_archive'
UNION ALL
SELECT 'gdelt_cleaned_rows', CAST(NULL AS BIGINT), COUNT(*), 'INFO' FROM cleaned_gdelt_events
UNION ALL
SELECT 'gdelt_daily_feature_days', CAST(NULL AS BIGINT), COUNT(*), 'INFO' FROM daily_gdelt_features;
__ASSET_EOF__

In [3]:
# --- PySpark feature builder and validator ---
cat > "$ASSETS/build_features.py" <<'__ASSET_EOF__'
#!/usr/bin/env python3
"""Task 2 / stage spark_features: distributed cleaning-aware feature, target and split builder.

spark-submit build_features.py --db <hive db> --out <hdfs run dir> --local-json features_summary.json

Reads the Hive tables created by the Hive stages (cleaned_prices, daily_gdelt_features and,
optionally, out_of_scope_prices), builds per-ticker rolling features with Spark window
functions, joins market-wide and (lagged) GDELT features, creates the targets, applies the
chronological split and boundary rules, and registers the results as Hive tables so that
Hive and Spark both see them. Nothing is collected to the driver except small summaries.

Prediction-date rule: every feature at trade_date t uses data up to the close of t.
GDELT enters through DATEADDED + GDELT_LAG_DAYS, so day t only sees news published before t.
Targets (label, future_return_5d, future_volatility_5d) look 5 trading sessions ahead.
A row is kept in a split only when its whole target window ends inside the same period
(training <= 2015-12-31, validation <= 2016-12-31, testing <= 2017-12-31).
Rows without a complete target are excluded and counted, never filled.
"""
import argparse
import json

from pyspark.sql import SparkSession, Window, functions as F

ap = argparse.ArgumentParser()
ap.add_argument("--db", required=True)
ap.add_argument("--out", required=True)
ap.add_argument("--local-json", default="features_summary.json")
ap.add_argument("--use-2018-labels", type=int, default=0,
                help="1 = compute late-2017 targets from 2018 rows (outside the brief's window)")
ap.add_argument("--min-vol-window", type=int, default=20,
                help="minimum return observations required for volatility_20d")
ap.add_argument("--gdelt-lag-days", type=int, default=1)
a = ap.parse_args()

H = 5  # prediction horizon in trading sessions
PERIODS = {  # split -> (start, end)
    "training": ("2013-01-01", "2015-12-31"),
    "validation": ("2016-01-01", "2016-12-31"),
    "testing": ("2017-01-01", "2017-12-31"),
}
CORE = ["open_price", "high_price", "low_price", "close_price", "volume",
        "daily_return", "return_5d", "volatility_20d", "volume_change_1d",
        "intraday_range", "volume_ratio_20d", "mkt_return_1d", "mkt_volatility_20d"]
GDELT_COLS = ["gdelt_event_count", "gdelt_avg_tone", "gdelt_avg_goldstein",
              "gdelt_total_mentions", "gdelt_available"]
TARGETS = ["label", "future_return_5d", "future_volatility_5d"]

spark = (SparkSession.builder.appName("task2_features_" + a.db)
         .enableHiveSupport().getOrCreate())
spark.sparkContext.setLogLevel("WARN")
db = a.db

# ---------------------------------------------------------------- 1. read Hive tables
prices = spark.table(db + ".cleaned_prices").select(
    F.trim("ticker").alias("ticker"),
    F.col("trade_date").cast("date").alias("trade_date"),
    F.col("open_price").cast("double"), F.col("high_price").cast("double"),
    F.col("low_price").cast("double"), F.col("close_price").cast("double"),
    F.col("volume").cast("long"))

if a.use_2018_labels:
    ctx = (spark.table(db + ".out_of_scope_prices").select(
        F.trim("name").alias("ticker"),
        F.to_date("date", "yyyy-MM-dd").alias("trade_date"),
        F.col("open").cast("double").alias("open_price"),
        F.col("high").cast("double").alias("high_price"),
        F.col("low").cast("double").alias("low_price"),
        F.col("close").cast("double").alias("close_price"),
        F.col("volume").cast("long").alias("volume"))
        .filter(F.col("ticker").isNotNull() & F.col("trade_date").isNotNull()
                & F.col("close_price").isNotNull()))
    prices = prices.unionByName(ctx)

prices = prices.dropDuplicates(["ticker", "trade_date"])

# ---------------------------------------------------------------- 2. per-ticker windows
w = Window.partitionBy("ticker").orderBy("trade_date")
hist20 = w.rowsBetween(-19, 0)      # last 20 sessions including today
prev20 = w.rowsBetween(-20, -1)     # previous 20 sessions, excluding today
fwd = w.rowsBetween(1, H)           # the next H sessions (targets only)

f = (prices
     .withColumn("prev_close", F.lag("close_price", 1).over(w))
     .withColumn("close_lag_h", F.lag("close_price", H).over(w))
     .withColumn("prev_volume", F.lag("volume", 1).over(w))
     .withColumn("future_close", F.lead("close_price", H).over(w))
     .withColumn("future_trade_date", F.lead("trade_date", H).over(w))
     .withColumn("daily_return", F.when(F.col("prev_close") > 0,
                                        F.col("close_price") / F.col("prev_close") - 1))
     .withColumn("return_5d", F.when(F.col("close_lag_h") > 0,
                                     F.col("close_price") / F.col("close_lag_h") - 1))
     .withColumn("volume_change_1d", F.when(F.col("prev_volume") > 0,
                                            F.col("volume") / F.col("prev_volume") - 1))
     .withColumn("intraday_range", F.when(F.col("close_price") > 0,
                                          (F.col("high_price") - F.col("low_price")) / F.col("close_price")))
     .withColumn("n_ret_20", F.count("daily_return").over(hist20))
     .withColumn("volatility_20d", F.when(F.col("n_ret_20") >= a.min_vol_window,
                                          F.stddev_pop("daily_return").over(hist20)))
     .withColumn("n_vol_prev20", F.count("volume").over(prev20))
     .withColumn("avg_vol_prev20", F.avg("volume").over(prev20))
     .withColumn("volume_ratio_20d", F.when((F.col("n_vol_prev20") >= 20) & (F.col("avg_vol_prev20") > 0),
                                            F.col("volume") / F.col("avg_vol_prev20")))
     .withColumn("n_fwd_ret", F.count("daily_return").over(fwd))
     .withColumn("future_volatility_5d", F.when(F.col("n_fwd_ret") == H,
                                                F.stddev_pop("daily_return").over(fwd)))
     .withColumn("future_return_5d", F.when((F.col("future_close").isNotNull()) & (F.col("close_price") > 0),
                                            F.col("future_close") / F.col("close_price") - 1))
     .withColumn("label", F.when(F.col("future_close").isNotNull(),
                                 F.when(F.col("future_close") > F.col("close_price"), 1).otherwise(0))))

# ---------------------------------------------------------------- 3. market-wide features (same-day cross-section)
mkt = f.groupBy("trade_date").agg(F.avg("daily_return").alias("mkt_return_1d"),
                                  F.avg("volatility_20d").alias("mkt_volatility_20d"))
f = f.join(F.broadcast(mkt), "trade_date", "left")

# ---------------------------------------------------------------- 4. GDELT integration (availability date + lag)
g = (spark.table(db + ".daily_gdelt_features")
     .select(F.date_add("available_date", a.gdelt_lag_days).alias("trade_date"),
             F.col("event_count").alias("gdelt_event_count"),
             F.col("avg_tone").alias("gdelt_avg_tone"),
             F.col("avg_goldstein").alias("gdelt_avg_goldstein"),
             F.col("total_mentions").alias("gdelt_total_mentions")))
f = (f.join(F.broadcast(g), "trade_date", "left")
     .withColumn("gdelt_available", F.col("gdelt_event_count").isNotNull().cast("int")))

# ---------------------------------------------------------------- 5. chronological split + boundary rule
d = F.col("trade_date")
split = F.lit(None).cast("string")
period_end = F.lit(None).cast("date")
for name, (lo, hi) in PERIODS.items():
    cond = d.between(F.lit(lo).cast("date"), F.lit(hi).cast("date"))
    split = F.when(cond, F.lit(name)).otherwise(split)
    period_end = F.when(cond, F.lit(hi).cast("date")).otherwise(period_end)

fs = (f.withColumn("split", split).withColumn("period_end", period_end)
      .filter(F.col("split").isNotNull()))
has_target = (F.col("label").isNotNull() & F.col("future_volatility_5d").isNotNull()
              & F.col("future_return_5d").isNotNull() & F.col("future_trade_date").isNotNull())
crosses = has_target & (F.col("future_trade_date") > F.col("period_end")) & \
    ~((F.col("split") == "testing") & F.lit(bool(a.use_2018_labels)))
keep = has_target & ~crosses

split_report = {r["split"]: r.asDict() for r in fs.groupBy("split").agg(
    F.count("*").alias("period_rows"),
    F.sum((~has_target).cast("int")).alias("excluded_no_complete_target"),
    F.sum(crosses.cast("int")).alias("excluded_target_crosses_period_end"),
    F.sum(keep.cast("int")).alias("kept")).collect()}

out_cols = (["ticker", "trade_date"] + CORE + GDELT_COLS
            + ["future_close", "future_trade_date"] + TARGETS)
# ---------------------------------------------------------------- 6. write + register (Hive and Spark share the tables)
def register(df, table, path):
    cols = ", ".join("`%s` %s" % (n, t) for n, t in df.dtypes)
    spark.sql("DROP TABLE IF EXISTS %s.%s" % (db, table))
    spark.sql("CREATE EXTERNAL TABLE %s.%s (%s) STORED AS PARQUET LOCATION '%s'" % (db, table, cols, path))


summary = {"db": db, "params": vars(a), "horizon_sessions": H, "periods": PERIODS,
           "core_features": CORE, "gdelt_features": GDELT_COLS, "targets": TARGETS,
           "split_report": split_report, "splits": {}}

for name in PERIODS:
    fpath = "%s/features/%s" % (a.out, name)
    part = fs.filter((F.col("split") == name) & keep).select(*out_cols)
    part.write.mode("errorifexists").parquet(fpath)
    register(spark.read.parquet(fpath), "features_" + name, fpath)

    feat = spark.table("%s.features_%s" % (db, name))
    nulls = feat.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in CORE + GDELT_COLS[:-1]]).first().asDict()
    ready = feat.dropna(subset=CORE + TARGETS)
    for c in CORE + TARGETS:
        ready = ready.filter(~F.isnan(F.col(c).cast("double")))
    rpath = "%s/model_ready/%s" % (a.out, name)
    # the model-ready table keeps ids, core features, GDELT columns and targets only;
    # future_close / future_trade_date stay in the features table for audit, never as features.
    ready.select(["ticker", "trade_date"] + CORE + GDELT_COLS + TARGETS).write.mode("errorifexists").parquet(rpath)
    register(spark.read.parquet(rpath), "model_ready_" + name, rpath)

    mr = spark.table("%s.model_ready_%s" % (db, name))
    st = mr.agg(F.count("*").alias("rows"), F.min("trade_date").alias("first_date"),
                F.max("trade_date").alias("last_date"),
                F.countDistinct("ticker").alias("tickers"),
                F.sum("label").alias("label_1"),
                F.sum("gdelt_available").alias("rows_with_gdelt")).first().asDict()
    summary["splits"][name] = {
        "features_rows": feat.count(), "null_counts_before_dropna": nulls,
        "model_ready": {k: (str(v) if k.endswith("_date") else v) for k, v in st.items()},
        "dropped_for_missing_values": feat.count() - st["rows"]}
    print(name, json.dumps(summary["splits"][name], default=str))

with open(a.local_json, "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
print("split_report", json.dumps(split_report, default=str))
spark.stop()
__ASSET_EOF__
cat > "$ASSETS/validate_features.py" <<'__ASSET_EOF__'
#!/usr/bin/env python3
"""Task 2 / stage spark_validate: re-check the prepared tables from Hive (read through Spark).

spark-submit validate_features.py --db <hive db> --summary-json features_summary.json \
                                  --local-json validation.json [--use-2018-labels 0]

Checks (each reported PASS/FAIL; exit code 1 if any FAIL):
  split dates stay inside their period and the periods do not overlap (no shuffling across periods)
  target windows end inside the period (boundary rule) for training and validation, and testing
  no duplicate (ticker, trade_date) rows
  model-ready tables contain no nulls / NaN / infinities in features or targets, labels are 0 or 1
  leakage columns (future_close, future_trade_date) are absent from the model-ready tables
  row accounting: Hive split rows = kept rows + excluded rows for every split
"""
import argparse
import json
import sys

from pyspark.sql import SparkSession, functions as F

ap = argparse.ArgumentParser()
ap.add_argument("--db", required=True)
ap.add_argument("--summary-json", required=True)
ap.add_argument("--local-json", default="validation.json")
ap.add_argument("--use-2018-labels", type=int, default=0)
a = ap.parse_args()

PERIODS = {"training": ("2013-01-01", "2015-12-31"),
           "validation": ("2016-01-01", "2016-12-31"),
           "testing": ("2017-01-01", "2017-12-31")}
LEAK = ["future_close", "future_trade_date"]

spark = SparkSession.builder.appName("task2_validate_" + a.db).enableHiveSupport().getOrCreate()
spark.sparkContext.setLogLevel("WARN")
summ = json.load(open(a.summary_json))
core, gd, targets = summ["core_features"], summ["gdelt_features"], summ["targets"]
checks = []


def add(name, ok, detail):
    checks.append({"check": name, "status": "PASS" if ok else "FAIL", "detail": detail})
    print("%-52s %s  %s" % (name, "PASS" if ok else "FAIL", detail))


ranges = {}
for name, (lo, hi) in PERIODS.items():
    feat = spark.table("%s.features_%s" % (a.db, name))
    ready = spark.table("%s.model_ready_%s" % (a.db, name))
    r = feat.agg(F.min("trade_date"), F.max("trade_date"), F.max("future_trade_date"),
                 F.count("*")).first()
    ranges[name] = (str(r[0]), str(r[1]))
    add("%s dates inside period" % name, str(r[0]) >= lo and str(r[1]) <= hi,
        "%s..%s expected within %s..%s" % (r[0], r[1], lo, hi))
    limit = "2099-12-31" if (name == "testing" and a.use_2018_labels) else hi
    add("%s target window ends inside period" % name, r[2] is None or str(r[2]) <= limit,
        "max future_trade_date=%s limit=%s" % (r[2], limit))
    dup = feat.groupBy("ticker", "trade_date").count().filter("count > 1").count()
    add("%s no duplicate ticker/date" % name, dup == 0, "duplicate keys=%d" % dup)
    bad = ready.agg(*[F.sum((F.col(c).isNull() | F.isnan(F.col(c).cast("double"))
                             | F.col(c).cast("double").isin(float("inf"), float("-inf"))).cast("int")).alias(c)
                      for c in core + targets]).first().asDict()
    total_bad = sum(v or 0 for v in bad.values())
    add("%s no null/NaN/inf in features and targets" % name, total_bad == 0,
        "bad cells=%d" % total_bad)
    lab = ready.filter(~F.col("label").isin(0, 1)).count()
    add("%s label is 0/1" % name, lab == 0, "invalid labels=%d" % lab)
    leaked = [c for c in LEAK if c in ready.columns]
    add("%s no leakage columns in model-ready" % name, not leaked, "found=%s" % leaked)
    hive_rows = spark.table("%s.%s_prices" % (a.db, name)).count()
    rep = summ["split_report"].get(name, {})
    acc = (rep.get("kept", 0) + rep.get("excluded_no_complete_target", 0)
           + rep.get("excluded_target_crosses_period_end", 0))
    add("%s Hive split rows = kept + excluded" % name, acc == hive_rows,
        "hive=%d kept=%d no_target=%d crosses_end=%d" % (
            hive_rows, rep.get("kept", 0), rep.get("excluded_no_complete_target", 0),
            rep.get("excluded_target_crosses_period_end", 0)))

order = [ranges[n] for n in ("training", "validation", "testing")]
add("periods are chronological and disjoint", order[0][1] < order[1][0] and order[1][1] < order[2][0],
    str(order))

res = {"db": a.db, "checks": checks, "failed": [c["check"] for c in checks if c["status"] == "FAIL"]}
with open(a.local_json, "w") as fh:
    json.dump(res, fh, indent=2)
print("FAILED CHECKS:", res["failed"] or "none")
spark.stop()
sys.exit(1 if res["failed"] else 0)
__ASSET_EOF__

In [4]:
# --- run summary and repetition aggregator (plain Python 3) ---
cat > "$ASSETS/summarize_run.py" <<'__ASSET_EOF__'
#!/usr/bin/env python3
"""Summarise one pipeline run: stage timings, resource peaks, row/byte counts and checks.

python3 summarize_run.py <run_dir>      (plain python3; no pandas needed)
Reads  : timings.csv, resources_raw.log, disk.csv, reconcile.tsv, inputs.txt,
         features_summary.json, validation.json, run_manifest.json
Writes : run_summary.json and prints a readable table.
"""
import csv
import datetime as dt
import json
import os
import re
import sys

run = sys.argv[1]
P = lambda n: os.path.join(run, n)


def load_json(name):
    try:
        return json.load(open(P(name)))
    except Exception:
        return {}


def iso(s):
    return dt.datetime.strptime(s, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=dt.timezone.utc)


# ---- stage timings
stages = []
if os.path.exists(P("timings.csv")):
    stages = list(csv.DictReader(open(P("timings.csv"))))
timed = [s for s in stages if s["exit_code"] not in ("SKIPPED",)]
failed = [s["stage"] for s in timed if s["exit_code"] != "0"]
skipped = [s["stage"] for s in stages if s["exit_code"] == "SKIPPED"]
secs = {s["stage"]: float(s["seconds"]) for s in timed}
wall = None
if timed:
    wall = (iso(timed[-1]["end_utc"]) - iso(timed[0]["start_utc"])).total_seconds()

# ---- reconciliation and inputs
recon = {}
if os.path.exists(P("reconcile.tsv")):
    for line in open(P("reconcile.tsv")):
        f = line.rstrip("\n").split("\t")
        if len(f) >= 4:
            recon[f[0]] = {"expected": f[1], "actual": f[2], "status": f[3]}
inputs = {}
if os.path.exists(P("inputs.txt")):
    for line in open(P("inputs.txt")):
        if "=" in line:
            k, v = line.strip().split("=", 1)
            inputs[k] = v

# ---- resources (best effort parsing of the sampler log)
res = {"samples": 0, "peak_primary_mem_used_mb": None, "peak_yarn_allocated_mb": None, "datanodes": {}}
if os.path.exists(P("resources_raw.log")):
    blocks = open(P("resources_raw.log"), errors="replace").read().split("### epoch=")[1:]
    res["samples"] = len(blocks)
    for b in blocks:
        m = re.search(r"MEM used_mb=(\d+)", b)
        if m:
            v = int(m.group(1))
            res["peak_primary_mem_used_mb"] = max(res["peak_primary_mem_used_mb"] or 0, v)
        alloc = [int(x) for x in re.findall(r"Allocated Resources\s*:\s*<memory:(\d+)", b)]
        if alloc:
            res["peak_yarn_allocated_mb"] = max(res["peak_yarn_allocated_mb"] or 0, sum(alloc))
        node = None
        for line in b.splitlines():
            if line.startswith("Name:"):
                node = line.split()[1]
                res["datanodes"].setdefault(node, {"peak_dfs_used_gb": 0, "peak_non_dfs_used_gb": 0,
                                                   "min_dfs_remaining_gb": None})
            elif node:
                for key, tag in (("DFS Used:", "peak_dfs_used_gb"), ("Non DFS Used:", "peak_non_dfs_used_gb"),
                                 ("DFS Remaining:", "min_dfs_remaining_gb")):
                    if line.startswith(key):
                        gb = int(line.split()[len(key.split())]) / 1024 ** 3
                        d = res["datanodes"][node]
                        if tag.startswith("min"):
                            d[tag] = round(gb if d[tag] is None else min(d[tag], gb), 2)
                        else:
                            d[tag] = round(max(d[tag], gb), 2)

# ---- disk: HDFS bytes written by this run
out_bytes = out_repl = 0
if os.path.exists(P("disk.csv")):
    for row in csv.DictReader(open(P("disk.csv"))):
        try:
            out_bytes = max(out_bytes, int(row["run_dir_bytes"]))
            out_repl = max(out_repl, int(row["run_dir_bytes_with_replicas"]))
        except Exception:
            pass

feat, val, man = load_json("features_summary.json"), load_json("validation.json"), load_json("run_manifest.json")
summary = {
    "run_name": man.get("run_name") or os.path.basename(run.rstrip("/")),
    "profile": man.get("profile"), "config": man.get("config"), "rep": man.get("rep"),
    "status": "FAILED" if failed or val.get("failed") or any(
        v["status"] == "FAIL" for v in recon.values()) else ("INCOMPLETE" if skipped else "OK"),
    "failed_stages": failed, "skipped_stages": skipped,
    "failed_checks": val.get("failed", []),
    "failed_reconciliations": [k for k, v in recon.items() if v["status"] == "FAIL"],
    "stage_seconds": secs, "sum_stage_seconds": round(sum(secs.values()), 1),
    "wall_clock_seconds": wall,
    "inputs": inputs, "resource_settings": man.get("resource_settings", {}),
    "reconciliation": recon,
    "split_report": feat.get("split_report", {}),
    "model_ready": {k: v.get("model_ready") for k, v in feat.get("splits", {}).items()},
    "resources": res, "run_output_bytes": out_bytes, "run_output_bytes_with_replicas": out_repl,
    "app_ids": {s["stage"]: s["app_ids"] for s in timed},
}
json.dump(summary, open(P("run_summary.json"), "w"), indent=2, default=str)

print("RUN", summary["run_name"], "->", summary["status"])
print("%-22s %-10s %10s %6s  %s" % ("stage", "engine", "seconds", "exit", "ids"))
for s in stages:
    print("%-22s %-10s %10s %6s  %s" % (s["stage"], s["engine"], s["seconds"], s["exit_code"], s["app_ids"]))
print("sum of stage seconds: %.1f   wall clock first start to last end: %s" % (
    summary["sum_stage_seconds"], "%.1f" % wall if wall else "n/a"))
print("peak primary memory used (MB):", res["peak_primary_mem_used_mb"],
      "| peak YARN allocated (MB):", res["peak_yarn_allocated_mb"])
for n, d in res["datanodes"].items():
    print("  datanode", n, d)
print("run output on HDFS: %.2f GiB (%.2f GiB with replicas)" % (out_bytes / 1024 ** 3, out_repl / 1024 ** 3))
for k, v in recon.items():
    print("  %-50s expected=%-12s actual=%-12s %s" % (k, v["expected"], v["actual"], v["status"]))
if summary["failed_stages"] or summary["failed_checks"] or summary["failed_reconciliations"]:
    print("FAILURES:", summary["failed_stages"], summary["failed_checks"], summary["failed_reconciliations"])
__ASSET_EOF__
cat > "$ASSETS/aggregate_runs.py" <<'__ASSET_EOF__'
#!/usr/bin/env python3
"""Aggregate repeated runs into the Section 6 tables.

python3 aggregate_runs.py <runs_root>      (default ~/task2_runs)
For every (profile, config): all repetitions, median and range of each stage and of the total,
resource peaks and failures. Also compares the OUTPUT FINGERPRINT (reconciliation counts and
split row counts) across all runs of the same profile, so identical inputs give identical outputs
whatever the configuration.
Writes aggregate_runs.csv and aggregate_report.md in <runs_root>.
"""
import csv
import glob
import json
import os
import statistics as st
import sys
from collections import defaultdict

root = os.path.expanduser(sys.argv[1] if len(sys.argv) > 1 else "~/task2_runs")
runs = []
for p in sorted(glob.glob(os.path.join(root, "*", "run_summary.json"))):
    try:
        runs.append(json.load(open(p)))
    except Exception as e:
        print("skip", p, e)
if not runs:
    sys.exit("no run_summary.json files under " + root)

groups = defaultdict(list)
for r in runs:
    groups[(r["profile"], r["config"])].append(r)

lines, rows = [], []
for (profile, config), rs in sorted(groups.items()):
    ok = [r for r in rs if r["status"] == "OK"]
    lines.append("### %s / %s  (%d runs, %d OK)" % (profile, config, len(rs), len(ok)))
    lines.append("")
    lines.append("| rep | status | total stage s | wall s | peak primary MB | peak YARN MB | output GiB (x2) |")
    lines.append("|---|---|---|---|---|---|---|")
    for r in sorted(rs, key=lambda x: x["rep"] or 0):
        lines.append("| %s | %s | %s | %s | %s | %s | %.2f |" % (
            r["rep"], r["status"], r["sum_stage_seconds"],
            "%.0f" % r["wall_clock_seconds"] if r["wall_clock_seconds"] else "n/a",
            r["resources"]["peak_primary_mem_used_mb"], r["resources"]["peak_yarn_allocated_mb"],
            r["run_output_bytes_with_replicas"] / 1024 ** 3))
    for x in rs:
        if x["status"] != "OK":
            lines.append("- FAILURE rep %s: stages=%s checks=%s reconciliations=%s" % (
                x["rep"], x["failed_stages"], x["failed_checks"], x["failed_reconciliations"]))
    lines.append("")
    stages = []
    for r in ok:
        for s in r["stage_seconds"]:
            if s not in stages:
                stages.append(s)
    lines.append("| stage | " + " | ".join("rep%s" % r["rep"] for r in ok) + " | median | min | max |")
    lines.append("|---|" + "---|" * (len(ok) + 3))
    for s in stages + ["__total__"]:
        vals = [(r["sum_stage_seconds"] if s == "__total__" else r["stage_seconds"].get(s)) for r in ok]
        nums = [v for v in vals if v is not None]
        if not nums:
            continue
        med, lo, hi = st.median(nums), min(nums), max(nums)
        label = "TOTAL (sum of stages)" if s == "__total__" else s
        lines.append("| %s | %s | %.1f | %.1f | %.1f |" % (
            label, " | ".join("%.1f" % v if v is not None else "-" for v in vals), med, lo, hi))
        rows.append({"profile": profile, "config": config, "stage": label, "runs": len(nums),
                     "median_s": round(med, 1), "min_s": lo, "max_s": hi,
                     "values_s": ";".join("%.1f" % v for v in nums)})
    lines.append("")

# output equivalence across all runs of a profile (any configuration, any repetition)
lines.append("### Output fingerprint check (identical inputs must give identical outputs)")
lines.append("")
by_profile = defaultdict(list)
for r in runs:
    if r["status"] == "OK":
        fp = {k: v["actual"] for k, v in r["reconciliation"].items()}
        fp.update({"kept_" + k: v.get("kept") for k, v in r["split_report"].items()})
        fp.update({"model_ready_rows_" + k: (v or {}).get("rows") for k, v in r["model_ready"].items()})
        by_profile[r["profile"]].append((r["run_name"], fp))
for profile, items in sorted(by_profile.items()):
    base_name, base = items[0]
    diffs = [(n, k, base.get(k), fp.get(k)) for n, fp in items[1:] for k in base if fp.get(k) != base.get(k)]
    lines.append("- %s: %d runs compared with %s -> %s" % (
        profile, len(items), base_name, "IDENTICAL" if not diffs else "DIFFERENCES: %s" % diffs[:10]))
lines.append("")

open(os.path.join(root, "aggregate_report.md"), "w").write("\n".join(lines))
with open(os.path.join(root, "aggregate_runs.csv"), "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=["profile", "config", "stage", "runs", "median_s", "min_s", "max_s", "values_s"])
    w.writeheader()
    w.writerows(rows)
print("\n".join(lines))
print("wrote", os.path.join(root, "aggregate_report.md"), "and aggregate_runs.csv")
__ASSET_EOF__

chmod +x "$ASSETS"/*.py
cd "$ASSETS" && sha256sum gdelt_zip_mapper.py sql/*.sql build_features.py validate_features.py summarize_run.py aggregate_runs.py | tee "$LOCAL_RUN/assets_sha256.txt"; cd - > /dev/null

2986a8519df72a194318c824fcf03b82a4569a426494ed1a3aff333deadc75e2  gdelt_zip_mapper.py


bbf711241b12e227b7a0f8df1a3bb948d711b137b6b6f03cde83c4663446f1a4  sql/01_create_db_raw_tables.sql


0a06b848faed6ff4501f97c8061d7bf29384dbf1250e29e15aff3c20b207eb9a  sql/02_clean_prices.sql


d63451e07971caaf26ee860a3fb5d3b49b76f23dd8abca46b7612b22f44b7765  sql/03_price_splits.sql


e7101a8d58214cec7830248f4c3778a5a4c7eef4ffc5609d850c22047c80afef  sql/04_clean_gdelt.sql


6c9c452e1fbda3e32bc65212aa7c1dd84823e19b9c7f6350c9afa3f326f07625  sql/05_reconcile.sql


4c48b6f5f9ccc67fdaae17aa369139f076fc4cb86b9f72aa0f8cd132ce4b6123  build_features.py


ea20e4e3c25639a56e13a0804552381e6288f3ad41de6376f8c00bea66c98010  validate_features.py


9303939b3ab4452505a4066e7e6b88616c1770892b25e1f0bad60daa194efca5  summarize_run.py


1a02110c0747f17f409a64e0050109eaa5d6629b4e5bd2152f92308ea94c7bd5  aggregate_runs.py


## 3. Helpers: stage timer, resource sampler, disk snapshots

In [5]:
now_s() { date +%s.%N; }

echo "timestamp,label,run_dir_bytes,run_dir_bytes_with_replicas" > "$LOCAL_RUN/disk.csv"
disk_snapshot() {   # disk_snapshot <label>  -> bytes under the run folder (and with replicas)
  local b r _x
  read -r b r _x < <(hdfs dfs -du -s "$OUT" 2>/dev/null || echo "0 0")
  echo "$(date -u +%FT%TZ),$1,${b:-0},${r:-0}" >> "$LOCAL_RUN/disk.csv"
}

echo "run_name,profile,config,rep,stage,engine,start_utc,end_utc,seconds,exit_code,app_ids" > "$LOCAL_RUN/timings.csv"
run_stage() {       # run_stage <stage> <engine> <command...>   (logs, times and records the outcome)
  local name=$1 engine=$2; shift 2
  local log="$LOCAL_RUN/logs/${name}.log"
  if [[ -n "$FAILED" ]]; then
    echo "SKIPPED $name (earlier failure in: $FAILED)"
    echo "$RUN_NAME,$PROFILE,$CONFIG,$REP,$name,$engine,,,0,SKIPPED," >> "$LOCAL_RUN/timings.csv"
    return 0
  fi
  disk_snapshot "before_$name"
  local sv ev s e rc secs ids
  sv=$(date -u +%FT%TZ); s=$(now_s)
  "$@" > "$log" 2>&1
  rc=$?
  e=$(now_s); ev=$(date -u +%FT%TZ)
  secs=$(awk -v a="$s" -v b="$e" 'BEGIN{printf "%.1f", b-a}')
  ids=$(grep -ohE '(application|job)_[0-9]+_[0-9]+' "$log" | sort -u | paste -sd' ' -)
  echo "$RUN_NAME,$PROFILE,$CONFIG,$REP,$name,$engine,$sv,$ev,$secs,$rc,$ids" >> "$LOCAL_RUN/timings.csv"
  disk_snapshot "after_$name"
  printf '%-20s %-10s %9ss  exit=%s  %s\n' "$name" "$engine" "$secs" "$rc" "$ids"
  if [[ $rc -ne 0 ]]; then FAILED=$name; echo "--- last lines of $log"; tail -25 "$log"; fi
  return 0
}

start_sampler() {   # memory (primary), YARN allocation and per-DataNode HDFS/non-HDFS disk, every SAMPLE_SECS
  : > "$LOCAL_RUN/resources_raw.log"
  (
    while true; do
      {
        echo "### epoch=$(date +%s)"
        free -m | awk '/^Mem:/ {print "MEM used_mb=" $3 " avail_mb=" $7}'
        hdfs dfsadmin -report 2>/dev/null | grep -E '^(Name|DFS Used|Non DFS Used|DFS Remaining):'
        yarn node -list -showDetails 2>/dev/null | grep -E 'Allocated Resources'
      } >> "$LOCAL_RUN/resources_raw.log" 2>/dev/null
      sleep "$SAMPLE_SECS"
    done
  ) &
  SAMPLER_PID=$!
  disown "$SAMPLER_PID" 2>/dev/null
}
stop_sampler() { if [[ -n "${SAMPLER_PID:-}" ]]; then kill "$SAMPLER_PID" 2>/dev/null; fi; SAMPLER_PID=; }
echo "helpers ready"

helpers ready


## 4. Pre-flight checks and run manifest (not timed)

In [6]:
ok=1
for c in hdfs hive hadoop spark-submit python3; do command -v $c > /dev/null || { echo "MISSING command: $c"; ok=0; }; done
[[ -f "$STREAMING_JAR" ]] || { echo "MISSING streaming jar (set STREAMING_JAR)"; ok=0; }
[[ "$CONFIG" != invalid && ( "$PROFILE" == dev || "$PROFILE" == full ) ]] || { echo "PROFILE must be dev|full and CONFIG reference|alt"; ok=0; }

# ---- discover the staged inputs (originals, written by the staging notebook) ----
CANDIDATES=$(hdfs dfs -ls -R "$RAW_PRICES_ROOT" 2>/dev/null | awk '$NF ~ /\.csv$/ {print $NF}')
echo "price CSV candidates:"; echo "$CANDIDATES"
PRICES_FILE=${RAW_PRICES_FILE:-$(echo "$CANDIDATES" | head -1)}
[[ -n "$PRICES_FILE" ]] || { echo "no price CSV under $RAW_PRICES_ROOT (set RAW_PRICES_FILE)"; ok=0; }
[[ $(echo "$CANDIDATES" | grep -c .) -le 1 ]] || echo "WARNING: several price CSVs found; using $PRICES_FILE (override with RAW_PRICES_FILE)"
hdfs dfs -ls -R "$RAW_GDELT_DIR" 2>/dev/null | awk '$NF ~ /\.zip$/ {print $5, $NF}' | sort -k2 > "$LOCAL_RUN/gdelt_all.txt"
echo "GDELT archives staged: $(wc -l < "$LOCAL_RUN/gdelt_all.txt")"
[[ -s "$LOCAL_RUN/gdelt_all.txt" ]] || { echo "no GDELT zips under $RAW_GDELT_DIR"; ok=0; }
[[ $(awk '{n=split($2,a,"/"); print a[n]}' "$LOCAL_RUN/gdelt_all.txt" | sort | uniq -d | wc -l) -eq 0 ]] || echo "WARNING: duplicate archive names across versions in $RAW_GDELT_DIR"
hdfs dfs -test -e "$OUT" && { echo "output folder already exists: $OUT"; ok=0; }
echo "HDFS capacity:"; hdfs dfs -df -h / 2>/dev/null | tail -2
if [[ $ok -eq 1 ]]; then echo "PRE-FLIGHT PASS"; else
  echo "PRE-FLIGHT FAIL"; FAILED=preflight; t=$(date -u +%FT%TZ)
  echo "$RUN_NAME,$PROFILE,$CONFIG,$REP,preflight,check,$t,$t,0.0,1," >> "$LOCAL_RUN/timings.csv"
fi

# ---- run manifest (parameters, settings, versions) ----
{ hadoop version 2>&1 | head -1; hive --version 2>&1 | grep -i hive | head -1; spark-submit --version 2>&1 | grep -iE "version|scala" | head -2; python3 --version; } > "$LOCAL_RUN/software_versions.txt" 2>&1
export PRICES_FILE SHUFFLE_PARTS LINES_PER_MAP
python3 - <<'PY'
import json, os, datetime
e = os.environ
json.dump({
  "run_name": e["RUN_NAME"], "profile": e["PROFILE"], "config": e["CONFIG"], "rep": int(e["REP"]),
  "run_id": e["RUN_ID"], "started_utc": datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
  "hive_db": e["DB"], "hdfs_output": e["OUT"], "profile_dir": e["PROFILE_DIR"],
  "inputs": {"prices_file": e["PRICES_FILE"], "gdelt_dir": e["RAW_GDELT_DIR"]},
  "resource_settings": {
    "spark": "2 executors x 2 cores x 6g heap + 2g overhead, driver 2g, dynamic allocation off",
    "spark.sql.shuffle.partitions": int(e["SHUFFLE_PARTS"]),
    "streaming_lines_per_map": int(e["LINES_PER_MAP"]),
    "dev_tickers": int(e["DEV_TICKERS"]), "dev_gdelt_archives": int(e["DEV_GDELT_ARCHIVES"]),
    "use_2018_labels": int(e["USE_2018_LABELS"]), "sample_secs": int(e["SAMPLE_SECS"])},
  "software_versions_file": "software_versions.txt", "assets_checksums_file": "assets_sha256.txt"},
  open(e["LOCAL_RUN"] + "/run_manifest.json", "w"), indent=2)
print("run_manifest.json written")
PY

price CSV candidates:


/corpus/raw/prices/v1_2013-2017/acq_2026-10-28/all_stocks_5yr.csv


GDELT archives staged: 150


HDFS capacity:


Filesystem                                   Size    Used  Available  Use%


hdfs://ip-172-31-30-82.ec2.internal:8020  188.9 G  23.1 G    152.0 G   12%


PRE-FLIGHT PASS


run_manifest.json written


## 5. Frozen input profile (not timed, built once per profile)
* **full** – every staged price file and all staged GDELT archives.
* **dev** – the first `DEV_TICKERS` tickers (alphabetical) of the price file and the first `DEV_GDELT_ARCHIVES` archives (by name).

The dev inputs are physically smaller (fewer bytes and rows), so dev→full is genuine input growth. The lists are stored in HDFS and
copied to the run evidence.

In [7]:
if [[ -z "$FAILED" ]]; then
  hdfs dfs -mkdir -p "$PROFILE_DIR"
  # ---- GDELT archive list ----
  if [[ "$PROFILE" == full ]]; then cp "$LOCAL_RUN/gdelt_all.txt" "$LOCAL_RUN/gdelt_profile.txt"
  else head -n "$DEV_GDELT_ARCHIVES" "$LOCAL_RUN/gdelt_all.txt" > "$LOCAL_RUN/gdelt_profile.txt"; fi
  awk '{print $2}' "$LOCAL_RUN/gdelt_profile.txt" > "$LOCAL_RUN/gdelt_paths.txt"
  hdfs dfs -put -f "$LOCAL_RUN/gdelt_paths.txt" "$PROFILE_DIR/gdelt_paths.txt"
  EXPECTED_ARCHIVES=$(wc -l < "$LOCAL_RUN/gdelt_paths.txt")

  # ---- price input folder ----
  if [[ "$PROFILE" == full ]]; then
    PRICES_IN=$(dirname "$PRICES_FILE")
    PRICES_BYTES=$(hdfs dfs -ls "$PRICES_FILE" | awk '{print $5}')
  else
    PRICES_IN=$PROFILE_DIR/prices
    if ! hdfs dfs -test -e "$PRICES_IN/prices_dev.csv"; then
      hdfs dfs -cat "$PRICES_FILE" | tr -d '\r' | awk -F, 'NR>1 {print $7}' | sort -u | head -n "$DEV_TICKERS" > "$LOCAL_RUN/dev_tickers.txt"
      hdfs dfs -cat "$PRICES_FILE" | tr -d '\r' | awk -F, 'NR==FNR {t[$1]=1; next} FNR==1 || ($7 in t)' "$LOCAL_RUN/dev_tickers.txt" - > "$LOCAL_RUN/prices_dev.csv"
      hdfs dfs -mkdir -p "$PRICES_IN"
      hdfs dfs -put -f "$LOCAL_RUN/prices_dev.csv" "$PRICES_IN/prices_dev.csv"
    fi
    PRICES_BYTES=$(hdfs dfs -ls "$PRICES_IN/prices_dev.csv" | awk '{print $5}')
  fi
  {
    echo "profile=$PROFILE"
    echo "prices_input_dir=$PRICES_IN"
    echo "prices_bytes=$PRICES_BYTES"
    echo "gdelt_archives=$EXPECTED_ARCHIVES"
    echo "gdelt_zip_bytes=$(awk '{s+=$1} END {printf "%d", s}' "$LOCAL_RUN/gdelt_profile.txt")"
    [[ "$PROFILE" == dev ]] && echo "dev_tickers=$(hdfs dfs -cat "$PRICES_IN/prices_dev.csv" | awk -F, 'NR>1 {print $7}' | sort -u | wc -l)"
  } > "$LOCAL_RUN/inputs.txt"
  cat "$LOCAL_RUN/inputs.txt"
fi

profile=full


prices_input_dir=/corpus/raw/prices/v1_2013-2017/acq_2026-10-28


prices_bytes=29580549


gdelt_archives=150


gdelt_zip_bytes=1925290170


## 6. Timed pipeline
Timing starts at the first stage below and ends after the last. Every stage runs in a fresh process (fresh Spark application, fresh Hive session).
If one stage fails, the later stages are skipped and the run is marked `FAILED` in the summary.

In [8]:
HIVEVARS=(--hivevar "DB=$DB" --hivevar "RAW_PRICES=$PRICES_IN" --hivevar "STREAM_OUT=$OUT/streaming/gdelt"
          --hivevar "OUT=$OUT" --hivevar "EXPECTED_ARCHIVES=$EXPECTED_ARCHIVES")
if [[ -z "$FAILED" ]]; then start_sampler; PIPELINE_START=$(date -u +%FT%TZ); echo "timed pipeline starts $PIPELINE_START"; fi

stage_gdelt_streaming() {
  hadoop jar "$STREAMING_JAR" \
    -D mapreduce.job.name="task2_${RUN_NAME}_gdelt" \
    -D mapreduce.input.lineinputformat.linespermap="$LINES_PER_MAP" \
    -D stream.map.input.ignoreKey=true \
    -D mapreduce.task.timeout=3600000 \
    -D mapreduce.output.fileoutputformat.compress=true \
    -D mapreduce.output.fileoutputformat.compress.codec=org.apache.hadoop.io.compress.GzipCodec \
    -files "$ASSETS/gdelt_zip_mapper.py" \
    -numReduceTasks 0 \
    -inputformat org.apache.hadoop.mapred.lib.NLineInputFormat \
    -mapper "python3 gdelt_zip_mapper.py" \
    -input "$PROFILE_DIR/gdelt_paths.txt" \
    -output "$OUT/streaming/gdelt"
}
run_stage gdelt_streaming streaming stage_gdelt_streaming

[1] 213685


timed pipeline starts 2026-11-20T11:42:57Z


gdelt_streaming      streaming      335.9s  exit=0  application_1795167376377_0050 job_1795167376377_0050


In [9]:
run_stage hive_raw_tables    hive hive "${HIVEVARS[@]}" -f "$ASSETS/sql/01_create_db_raw_tables.sql"
run_stage hive_clean_prices  hive hive "${HIVEVARS[@]}" -f "$ASSETS/sql/02_clean_prices.sql"
run_stage hive_price_splits  hive hive "${HIVEVARS[@]}" -f "$ASSETS/sql/03_price_splits.sql"
run_stage hive_clean_gdelt   hive hive "${HIVEVARS[@]}" -f "$ASSETS/sql/04_clean_gdelt.sql"

hive_raw_tables      hive            10.1s  exit=0  


hive_clean_prices    hive            35.5s  exit=0  application_1795167376377_0052


hive_price_splits    hive            27.9s  exit=0  application_1795167376377_0053


hive_clean_gdelt     hive           405.4s  exit=0  application_1795167376377_0054


In [10]:
stage_reconcile() {   # prints check<TAB>expected<TAB>actual<TAB>status; fails the stage on any FAIL
  hive -S "${HIVEVARS[@]}" -f "$ASSETS/sql/05_reconcile.sql" > "$LOCAL_RUN/reconcile.tsv" || return 1
  awk -F'\t' '$4 == "FAIL" {f = 1} END {exit f}' "$LOCAL_RUN/reconcile.tsv"
}
run_stage hive_reconcile hive stage_reconcile
if [[ -f "$LOCAL_RUN/reconcile.tsv" ]]; then column -t -s$'\t' "$LOCAL_RUN/reconcile.tsv" 2>/dev/null || cat "$LOCAL_RUN/reconcile.tsv"; fi
true

hive_reconcile       hive            80.0s  exit=0  


prices_raw_rows                              NULL      619040    INFO


prices_out_of_scope_rows                     NULL      13130     INFO


prices_raw_eq_cleaned_rejected_out_of_scope  619040    619040    PASS


prices_rejected_rows                         NULL      23        INFO


gdelt_structural_rejects_eq_audit            0         0         PASS


... (5 outputs left out)

gdelt_daily_feature_days                     NULL      150       INFO


gdelt_valid_eq_cleaned_plus_value_rejected   29042290  29042290  PASS


gdelt_cleaned_rows                           NULL      29042043  INFO


prices_cleaned_rows                          NULL      605887    INFO


In [11]:
run_stage spark_features pyspark spark-submit "${SPARK_ARGS[@]}" --name "task2_features_$RUN_NAME" \
    "$ASSETS/build_features.py" --db "$DB" --out "$OUT" --local-json "$LOCAL_RUN/features_summary.json" \
    --use-2018-labels "$USE_2018_LABELS"
run_stage spark_validate pyspark spark-submit "${SPARK_ARGS[@]}" --name "task2_validate_$RUN_NAME" \
    "$ASSETS/validate_features.py" --db "$DB" --summary-json "$LOCAL_RUN/features_summary.json" \
    --local-json "$LOCAL_RUN/validation.json" --use-2018-labels "$USE_2018_LABELS"

spark_features       pyspark         55.0s  exit=0  application_1795167376377_0056


spark_validate       pyspark         32.6s  exit=0  application_1795167376377_0057


**Optional Task 3 hook.** Set `TRAIN_CMD` to the command that trains and saves the model (for example a `spark-submit ... train.py --db $DB --out $OUT`), and it is timed as stage `train` inside the total.

In [12]:
if [[ -n "$TRAIN_CMD" ]]; then run_stage train pyspark bash -c "$TRAIN_CMD"; else echo "no TRAIN_CMD set; training stage not included"; fi
if [[ -n "${PIPELINE_START:-}" ]]; then PIPELINE_END=$(date -u +%FT%TZ); echo "timed pipeline ends $PIPELINE_END"; fi
stop_sampler
true

train                pyspark         88.9s  exit=0  application_1795167376377_0058


timed pipeline ends 2026-11-20T12:01:25Z


## 7. Run summary (not timed)
Stage timings, resource peaks, row and byte counts, reconciliation and validation results. Saved as `run_summary.json` / `run_summary.txt`.

In [13]:
cp -r "$ASSETS" "$LOCAL_RUN/assets_used" 2>/dev/null
cp "$PROFILE_DIR/gdelt_paths.txt" "$LOCAL_RUN/" 2>/dev/null
python3 "$ASSETS/summarize_run.py" "$LOCAL_RUN" | tee "$LOCAL_RUN/run_summary.txt"

RUN full_reference_rep2_20261120T114124Z -> OK


stage                  engine        seconds   exit  ids


gdelt_streaming        streaming       335.9      0  application_1795167376377_0050 job_1795167376377_0050


hive_raw_tables        hive             10.1      0  


hive_clean_prices      hive             35.5      0  application_1795167376377_0052


hive_price_splits      hive             27.9      0  application_1795167376377_0053


... (20 outputs left out)

  gdelt_daily_feature_days                           expected=NULL         actual=150          INFO


  gdelt_valid_eq_cleaned_plus_value_rejected         expected=29042290     actual=29042290     PASS


  gdelt_cleaned_rows                                 expected=NULL         actual=29042043     INFO


  prices_cleaned_rows                                expected=NULL         actual=605887       INFO


## 8. Clean-up and evidence (not timed)
Each full run writes tens of GB (the gzip Streaming output and the cleaned GDELT table). With replication 2 and six or more full runs,
keeping everything would exhaust HDFS, so the bulky intermediates are deleted with `-skipTrash` (trash would keep a second copy).
The reconciliation, rejected data, features, model-ready tables and all logs are kept. Set `KEEP_BULK=1` to keep everything.

In [14]:
if [[ "$KEEP_BULK" != "1" ]]; then
  for d in streaming cleaned/gdelt/events; do hdfs dfs -rm -r -f -skipTrash "$OUT/$d" > /dev/null 2>&1 && echo "deleted $OUT/$d"; done
fi
echo "kept on HDFS: $(hdfs dfs -du -s -h "$OUT" 2>/dev/null)"
hdfs dfs -df -h / 2>/dev/null | tail -2
if [[ -n "$BUCKET" ]]; then
  aws s3 cp --recursive "$LOCAL_RUN" "s3://$BUCKET/evidence/task2_runs/$RUN_NAME/" --only-show-errors && echo "uploaded evidence to s3://$BUCKET/evidence/task2_runs/$RUN_NAME/"
fi
echo "evidence folder: $LOCAL_RUN"

deleted /corpus/runs/full_reference_rep2_20261120T114124Z/streaming


deleted /corpus/runs/full_reference_rep2_20261120T114124Z/cleaned/gdelt/events


kept on HDFS: 127.7 M  255.3 M  /corpus/runs/full_reference_rep2_20261120T114124Z


Filesystem                                   Size    Used  Available  Use%


hdfs://ip-172-31-30-82.ec2.internal:8020  188.9 G  28.3 G    144.3 G   15%


uploaded evidence to s3://inf2006-bigdata/evidence/task2_runs/full_reference_rep2_20261120T114124Z/


evidence folder: /home/hadoop/task2_runs/full_reference_rep2_20261120T114124Z


## 9. Aggregate the repetitions (run after all runs are finished)
Prints, for every profile/configuration, all repetitions, the median and range of each stage and of the total, resource peaks, failures, and an
**output fingerprint check** that confirms identical inputs gave identical outputs whatever the configuration. Saved as `aggregate_report.md` and
`aggregate_runs.csv` under the runs folder; this is the table for Report Section 6.

In [15]:
python3 "$ASSETS/aggregate_runs.py" "$RUNS_ROOT"

### dev / reference  (3 runs, 3 OK)


| rep | status | total stage s | wall s | peak primary MB | peak YARN MB | output GiB (x2) |


|---|---|---|---|---|---|---|


| 1 | OK | 475.4 | 506 | 8026 | 24576 | 0.92 |


| 2 | OK | 475.3 | 506 | 8319 | 24576 | 0.92 |


| 3 | OK | 480.4 | 510 | 7936 | 24576 | 0.92 |


... (29 outputs left out)

### Output fingerprint check (identical inputs must give identical outputs)


- dev: 3 runs compared with dev_reference_rep1_20261120T103154Z -> IDENTICAL


- full: 2 runs compared with full_reference_rep1_20261120T112117Z -> IDENTICAL


wrote /home/hadoop/task2_runs/aggregate_report.md and aggregate_runs.csv
